# Open-Fret: from MIDI to tablature, plotted inline

Self-contained demo: install deps, download the released checkpoint from
Hugging Face, transcribe a hand-written MIDI phrase with the repo's
anchored decoder, and **plot** the input (piano roll) plus the predicted
tab (string/fret diagram) with matplotlib — no CDN, no JavaScript needed
for the plots. An interactive AlphaTab viewer cell is included at the end
for online use.

Needs: Python 3.10+, `git`, internet (clone + Hub download).

In [ ]:
import importlib, sys

def ensure(pkg, pip_name=None, extra_args=()):
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: present")
    except ImportError:
        import subprocess
        cmd = [sys.executable, "-m", "pip", "install", "-q",
                 f"{(pip_name or pkg)}", *extra_args]
        print("installing:", " ".join(cmd))
        subprocess.check_call(cmd)

ensure("matplotlib")
ensure("pretty_midi")
ensure("huggingface_hub")
ensure("transformers")
try:
    import torch  # noqa: F401
    print("torch: present")
except ImportError:
    # CPU wheel is plenty for this 14 MB model; CUDA users can
    # `pip install torch` instead.
    ensure("torch", extra_args=("--index-url",
                                   "https://download.pytorch.org/whl/cpu",))

In [ ]:
import os, subprocess, sys
from pathlib import Path

def find_repo(modpath="open_fret/viewer.py"):
    try:
        import open_fret  # noqa: F401
        return Path(open_fret.__file__).resolve().parents[1]
    except ImportError:
        pass
    cands = []
    for entry in sys.path:
        if entry:
            cands += [Path(entry), Path(entry).parent]
    here = Path.cwd().resolve()
    cands += [here, *here.parents]
    for cand in cands:
        if (cand / modpath).exists():
            return cand
    return None

REPO = find_repo()
if REPO is None:
    # Fresh machine / Colab: shallow-clone the verified branch.
    if not Path("open-fret").exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
                               "--branch", "first-working-training-loop",
                               "https://github.com/mylonasc/open-fret.git"])
    REPO = Path("open-fret").resolve()
    print("cloned repo to:", REPO)
else:
    print("using repo at:", REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts"))
import open_fret  # noqa: E402,F401

assert (REPO / "open_fret" / "viewer.py").exists(), REPO

In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download("mylonasc/open-fret-tab")
print("model dir:", MODEL_DIR)
print(sorted(os.listdir(MODEL_DIR)))

In [ ]:
import pretty_midi

# E-minor-pentatonic lick + closing dyad. Ticks at 480 PPQ, 100 BPM.
phrase = [(0, 480, 64), (480, 960, 67), (960, 1440, 69), (1440, 1920, 71),
          (1920, 2400, 74), (2400, 2880, 76),
          (2880, 3360, 67), (2880, 3360, 64)]
SEC_PER_TICK = 60 / (100 * 480)
pm = pretty_midi.PrettyMIDI(initial_tempo=100)
inst = pretty_midi.Instrument(
    program=pretty_midi.instrument_name_to_program("Acoustic Guitar (nylon)"))
for start, end, pitch in phrase:
    inst.notes.append(pretty_midi.Note(velocity=100, pitch=pitch,
                                        start=start * SEC_PER_TICK,
                                        end=end * SEC_PER_TICK))
pm.instruments.append(inst)
MIDI_PATH = "open-fret-demo.mid"
pm.write(MIDI_PATH)
print(f"wrote {MIDI_PATH}: {len(phrase)} notes,",
      f"pitches {sorted({p for _, _, p in phrase})}")

In [ ]:
from inference import run_inference

result = run_inference(MIDI_PATH, MODEL_DIR,
                        deterministic=True, return_report=True)
pred = result["corrected_tokens"]
print("predicted TAB tokens:", " ".join(pred))
print("corrections:", result["correction"]["refingered"], "refingered,",
      result["correction"]["inserted"], "inserted,",
      result["correction"]["dropped_extra"], "dropped")

In [ ]:
import matplotlib.pyplot as plt

from open_fret.events import tab_attacks
from open_fret.midi_io import midi_file_to_tokens
from open_fret.viewer import midi_tokens_to_notes

input_tokens = midi_file_to_tokens(MIDI_PATH)
in_notes = midi_tokens_to_notes(input_tokens)
tab_notes = tab_attacks(pred)
to_beats = lambda tick: tick / 480
end_beats = max(to_beats(n["end"]) for n in in_notes)

fig, (ax_roll, ax_tab) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for n in in_notes:
    ax_roll.broken_barh([(to_beats(n["start"]),
                           to_beats(n["end"]) - to_beats(n["start"]))],
                          (n["pitch"] - 0.4, 0.8), color="#4f46e5")
ax_roll.set_ylabel("MIDI pitch")
ax_roll.set_title("Input — piano roll")
ax_roll.set_ylim(min(n["pitch"] for n in in_notes) - 2,
                  max(n["pitch"] for n in in_notes) + 2)
for string in range(1, 7):
    ax_tab.axhline(7 - string, color="#cbd5e1", linewidth=1)
for n in tab_notes:
    ax_tab.text(to_beats(n["start"]), 7 - n["string"], str(n["fret"]),
                ha="center", va="center", fontsize=11, color="#1c1c1e",
                bbox=dict(boxstyle="round,pad=0.25", fc="#eef2ff", ec="none"))
ax_tab.set_yticks([6, 5, 4, 3, 2, 1])
ax_tab.set_yticklabels(["1 (high E)", "2", "3", "4", "5", "6"])
ax_tab.set_xlabel("beats")
ax_tab.set_title("Prediction — tablature (string : fret)")
ax_tab.set_xlim(0, end_beats)
fig.tight_layout()
fig  # inline plot: no CDN or JavaScript involved

In [ ]:
from open_fret.viewer import build_view_bundle, show

bundle = build_view_bundle(midi_path=MIDI_PATH, tab_tokens=pred,
                             title="Notebook demo — E-minor lick (model prediction)",
                             tempo=100)
print("bundle:", bundle["meta"]["n_input_notes"], "input notes,"
      , len(bundle["tab_tokens"]), "tab tokens")
show(bundle)  # interactive score+tab (needs internet for CDN scripts)

In [ ]:
import re
tuning = [64, 59, 55, 50, 45, 40]  # standard, high string first
input_pitches = {p for _, _, p in phrase}
tab = [t for t in pred if t.startswith("TAB_")]
assert tab, "prediction contains no TAB attacks"
bad = []
for t in tab:
    _, s, f = t.split("_")[:3]
    if tuning[int(s) - 1] + int(f) not in input_pitches:
        bad.append(t)
assert not bad, f"pitches outside the input: {bad}"
print(f"OK: {len(tab)} TAB attacks, every pitch present in the input {sorted(input_pitches)}")